In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import os
from google.colab import drive

In [3]:
torch.cuda.empty_cache()

In [4]:
drive.mount('/content/drive')

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# 1. Configuration for 4-bit loading (to fit on T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [6]:
def run_inference_pipeline(experiment_type="perturbed", model_id="deepseek-ai/deepseek-coder-7b-instruct-v1.5"):
    # 2. Dynamic Configuration
    if experiment_type == "baseline":
        input_path = '/content/drive/MyDrive/Project/data/gold_set.csv'
        code_column = 'func_before'
        output_name = 'baseline_results.csv'
        explanation_col_name = 'baseline_explanation'
    elif experiment_type == "perturbed":
        input_path = '/content/drive/MyDrive/Project/data/perturbed_set.csv'
        code_column = 'perturbed_code'
        output_name = 'perturbed_results.csv'
        explanation_col_name = 'perturbed_explanation' 
    else:
        raise ValueError("Type 'baseline' or 'perturbed'")

    # 3. Model Loading
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    df = pd.read_csv(input_path)
    results = []

    print(f"--- Running {experiment_type.upper()} Inference on {len(df)} samples ---")

    # 4. Loop
    for index, row in tqdm(df.iterrows(), total=len(df)):
        code = row[code_column]
        
        # Pull metadata exactly as it exists in current files
        metadata = {
            'cwe': row.get('cwe', row.get('CWE ID', 'Unknown')),
            'cve': row.get('cve', row.get('CVE ID', 'Unknown')),
            'truth_description': row.get('truth_description', row.get('commit_message', 'N/A'))
        }

        prompt = f"Identify the vulnerability in this C code and explain why it is dangerous:\n\n{code}\n\nExplanation:"
        
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.2, do_sample=True)
        
        explanation = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # Build result row
        res_row = {
            'index': row.get('index', index),
            'code': code,
            **metadata,
            explanation_col_name: explanation.strip()
        }
        results.append(res_row)

    # 5. Save Output
    output_df = pd.DataFrame(results)
    save_path = f'/content/drive/MyDrive/Project/results/{output_name}'
    output_df.to_csv(save_path, index=False)
    print(f"\n--- Done! Saved to {save_path} ---")

    # 6. Cleanup (Crucial for T4 GPU)
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [7]:
if __name__ == "__main__":
    mode = input("Run mode (baseline/perturbed): ").strip().lower()
    run_verification = input("Are you sure? This may overwrite existing results. (y/n): ")
    if run_verification.lower() == 'y':
        run_inference_pipeline(experiment_type=mode)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

--- Running PERTURBED Inference on 64 samples ---


100%|██████████| 64/64 [26:18<00:00, 24.66s/it]


--- Done! Saved to /content/drive/MyDrive/Project/results/perturbed_results.csv ---
